In [0]:
%run ../utils/config.py

In [0]:
from pyspark.sql import functions as F

# Läs in rådata från volymen
df = spark.read.csv(
    RAW_DATA_PATH,
    header=True,
    inferSchema=True,
)

# Antal rader och kolumner
print(f"Rader:    {df.count():,}")
print(f"Kolumner: {len(df.columns)}")

# Schema
df.printSchema()

# Beskrivande statistik för numeriska fält
df.describe().display()

# Räkna nullvärden per kolumn
null_counts = df.select([
    F.count(F.when(F.col(c).isNull() | (F.col(c).cast("string") == ""), c)).alias(c)
    for c in df.columns
])
null_counts.display()

# Hur många unika events finns det?
unique_events = df.select("Event name").distinct().count()
print(f"Unika events: {unique_events:,}")

# Åldersfördelning bland löpare
# Räknar ut ungefärlig ålder baserat på födelseår
df_age = df.withColumn(
    "approx_age",
    F.lit(2023) - F.col("Athlete year of birth").cast("int")
).filter(F.col("approx_age").isNotNull())

df_age.groupBy("approx_age") \
    .count() \
    .orderBy("approx_age") \
    .limit(20) \
    .display()

# Vilka länder är mest representerade?
df.groupBy("Athlete country") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .display()

# Fördelning av event-typer
df.groupBy("Event distance/length") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .display()